In [ ]:
"""
Gene Specificity Score Analysis
=================================
For each data type (neuron / non-neuron):
  1. Compute per-cell-type mean expression
  2. Compute gene specificity scores (expression / total across cell types)
  3. Assign each gene to its most specific cell type
  4. Merge with causal gene results
  5. Plot specificity heatmap for top causal genes
"""

import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib
matplotlib.rcParams["font.family"] = "DejaVu Sans"
import matplotlib.pyplot as plt


# ========================= Configuration =========================

# List of (label, h5ad_path, causal_genes_csv_path) tuples
DATA_CONFIGS = [
    ("neuron",    "", ""),   # (label, h5ad path, causal genes csv path)
    ("nonneuron", "", ""),
]

OUTPUT_DIR = "./results/"              # Output directory for CSV and figures


# ========================= Analysis =========================

for data_type, h5ad_path, causal_path in DATA_CONFIGS:
    print("=" * 60)
    print(f"[{data_type}] Computing gene specificity scores")
    print("=" * 60)

    adata = sc.read_h5ad(h5ad_path)
    causal = pd.read_csv(causal_path)

    # Normalize expression
    adata_norm = adata.copy()
    sc.pp.normalize_total(adata_norm, target_sum=1e4)
    sc.pp.log1p(adata_norm)

    all_ct = adata.obs["annotation"].unique().tolist()
    print(f"Cell subtypes: {len(all_ct)}")

    # Per-cell-type mean expression
    print("Computing mean expression matrix...")
    mean_expr_dict = {}
    for ct in all_ct:
        mask = (adata.obs["annotation"] == ct).values
        X_ct = adata_norm.X[mask, :]
        if hasattr(X_ct, "toarray"):
            mean_expr = np.array(X_ct.mean(axis=0)).flatten()
        else:
            mean_expr = X_ct.mean(axis=0).flatten()
        mean_expr_dict[ct] = mean_expr

    # Genes x cell_types mean expression matrix
    mean_expr_df = pd.DataFrame(mean_expr_dict, index=adata.var_names)

    # Specificity score: gene expression in cell type / sum across all cell types
    row_sums = mean_expr_df.sum(axis=1).replace(0, 1e-10)
    specificity_df = mean_expr_df.div(row_sums, axis=0)
    print(f"Specificity matrix: {specificity_df.shape}")

    # Assign each gene to its most specific cell type
    best_ct = specificity_df.idxmax(axis=1)
    best_spec = specificity_df.max(axis=1)

    gene_annotation = pd.DataFrame({
        "gene": adata.var_names,
        "best_celltype": best_ct.values,
        "specificity_score": best_spec.values,
    })

    # Merge with causal genes
    causal_annotated = causal.merge(gene_annotation, on="gene", how="left")

    # Summary
    print(f"\nCausal genes per cell type (top 10):")
    print(causal_annotated["best_celltype"].value_counts().head(10).to_string())

    sig_genes = causal_annotated[
        (causal_annotated["correlation"] > 0) &
        (causal_annotated["pvalue"] < 0.05 / len(causal_annotated))
    ]
    print(f"\nSignificant positive genes ({len(sig_genes)}) by cell type:")
    print(sig_genes["best_celltype"].value_counts().to_string())

    print(f"\nTop 30 genes:")
    print(causal_annotated[["gene", "correlation", "best_celltype", "specificity_score"]]
          .head(30).to_string(index=False))

    # Save tables
    causal_annotated.to_csv(f"{OUTPUT_DIR}/causal_genes_{data_type}_specificity.csv", index=False)
    specificity_df.to_csv(f"{OUTPUT_DIR}/specificity_matrix_{data_type}.csv")
    mean_expr_df.to_csv(f"{OUTPUT_DIR}/mean_expr_matrix_{data_type}.csv")

    # --- Heatmap: top 50 genes x all cell types ---
    top_genes = causal_annotated[causal_annotated["correlation"] > 0].head(50)
    top_genes_sorted = top_genes.sort_values(["best_celltype", "correlation"],
                                              ascending=[True, False])
    gene_order = top_genes_sorted["gene"].tolist()
    heatmap_data = specificity_df.loc[gene_order, all_ct]

    fig, ax = plt.subplots(figsize=(12, 18))
    im = ax.imshow(heatmap_data.values, cmap="YlOrRd", aspect="auto", vmin=0, vmax=0.3)

    ax.set_xticks(range(len(all_ct)))
    ax.set_xticklabels(all_ct, rotation=60, ha="right", fontsize=7)
    ax.set_yticks(range(len(gene_order)))

    y_labels = [f"{row['gene']} ({row['best_celltype'][:15]})"
                for _, row in top_genes_sorted.iterrows()]
    ax.set_yticklabels(y_labels, fontsize=6)

    cbar = plt.colorbar(im, ax=ax, shrink=0.4)
    cbar.set_label("Specificity Score", fontsize=10)

    # Cell type group dividers
    current_ct = ""
    for idx, (_, row) in enumerate(top_genes_sorted.iterrows()):
        if row["best_celltype"] != current_ct:
            if current_ct != "":
                ax.axhline(y=idx - 0.5, color="white", linewidth=2)
            current_ct = row["best_celltype"]

    ax.set_title(f"{data_type.capitalize()}: Top 50 Causal Genes\n"
                 f"Specificity Across Cell Types",
                 fontsize=13, fontweight="bold")

    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}/Fig_heatmap_specificity_{data_type}.pdf", dpi=300, bbox_inches="tight")
    plt.savefig(f"{OUTPUT_DIR}/Fig_heatmap_specificity_{data_type}.png", dpi=300, bbox_inches="tight")
    plt.close()

    print(f"\nSaved:")
    print(f"  causal_genes_{data_type}_specificity.csv")
    print(f"  specificity_matrix_{data_type}.csv")
    print(f"  mean_expr_matrix_{data_type}.csv")
    print(f"  Fig_heatmap_specificity_{data_type}.png")

print("\n" + "=" * 60)
print("Done.")